In [4]:
import pandas as pd
df = pd.read_csv('customer_shopping_behavior.csv')

In [5]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [7]:
df.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [8]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [9]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [10]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [11]:
df.columns =df.columns.str.lower()
df.columns =df.columns.str.replace(' ','_')
df=df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [12]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [13]:
#create a column age_group
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior' ]
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)

In [14]:
df[['age', 'age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [15]:
# create column purchase_frequency_days
frequency_mapping ={
    'Fortnightly':14,
    'Monthly':30,
    'Quaterly':90,
    'Weekly':7,
    'Annually':365,
    'Bi-weekly':14,
    'Every three months':90
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [16]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14.0,Fortnightly
1,14.0,Fortnightly
2,7.0,Weekly
3,7.0,Weekly
4,365.0,Annually
5,7.0,Weekly
6,NaN,Quarterly
7,7.0,Weekly
8,365.0,Annually
9,NaN,Quarterly


In [17]:
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [18]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [19]:
df =df.drop('promo_code_used', axis=1)

In [20]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [21]:
import sqlite3

# Create SQLite database
conn = sqlite3.connect('customer_behavior.db')

# Load DataFrame into SQLite
table_name = 'customer'

df.to_sql(
    table_name,
    conn,
    if_exists='replace',
    index=False
)

print(f"Data successfully loaded into table '{table_name}'.")

Data successfully loaded into table 'customer'.


In [22]:
pd.read_sql("SELECT * FROM customer LIMIT 5", conn)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14.0
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14.0
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7.0
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7.0
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365.0


In [23]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name
0,customer


In [24]:
pd.read_sql("""
SELECT gender,
       SUM(purchase_amount) AS total_revenue
FROM customer
GROUP BY gender;
""", conn)

,gender,total_revenue
0,Female,75191
1,Male,157890


In [25]:
pd.read_sql("""
SELECT gender,
       ROUND(AVG(purchase_amount),2) AS avg_purchase
FROM customer
GROUP BY gender
ORDER BY avg_purchase DESC;
""", conn)

,gender,avg_purchase
0,Female,60.25
1,Male,59.54


In [26]:
pd.read_sql("""
SELECT item_purchased,
       ROUND(AVG(review_rating),2) AS avg_rating
FROM customer
GROUP BY item_purchased
ORDER BY avg_rating DESC
LIMIT 10;
""", conn)

,item_purchased,avg_rating
0,Gloves,3.86
1,Sandals,3.84
2,Boots,3.82
3,Hat,3.80
4,T-shirt,3.78
5,Skirt,3.78
6,Handbag,3.78
7,Sweater,3.76
8,Sneakers,3.76
9,Jewelry,3.76


In [27]:
pd.read_sql("""
SELECT shipping_type,
       ROUND(AVG(purchase_amount),2) AS avg_purchase
FROM customer
GROUP BY shipping_type
ORDER BY avg_purchase DESC;
""", conn)

,shipping_type,avg_purchase
0,2-Day Shipping,60.73
1,Express,60.48
2,Free Shipping,60.41
3,Store Pickup,59.89
4,Next Day Air,58.63
5,Standard,58.46


In [28]:
pd.read_sql("""
SELECT subscription_status,
       COUNT(*) AS customers,
       ROUND(AVG(purchase_amount),2) AS avg_spend,
       SUM(purchase_amount) AS total_revenue
FROM customer
GROUP BY subscription_status;
""", conn)

,subscription_status,customers,avg_spend,total_revenue
0,No,2847,59.87,170436
1,Yes,1053,59.49,62645


In [29]:
pd.read_sql("""
SELECT discount_applied,
       COUNT(*) AS purchase_count,
       ROUND(
           COUNT(*) * 100.0 /
           (SELECT COUNT(*) FROM customer),
           2
       ) AS percentage
FROM customer
GROUP BY discount_applied;
""", conn)

,discount_applied,purchase_count,percentage
0,No,2223,57.0
1,Yes,1677,43.0


In [30]:
pd.read_sql("""
SELECT
CASE
    WHEN previous_purchases <= 10 THEN 'New'
    WHEN previous_purchases <= 30 THEN 'Returning'
    ELSE 'Loyal'
END AS customer_segment,
COUNT(*) AS customer_count
FROM customer
GROUP BY customer_segment;
""", conn)

,customer_segment,customer_count
0,Loyal,1549
1,New,784
2,Returning,1567


In [31]:
pd.read_sql("""
WITH ranked_products AS (
    SELECT
        category,
        item_purchased,
        COUNT(*) AS purchase_count,
        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM customer
    GROUP BY category, item_purchased
)
SELECT *
FROM ranked_products
WHERE rn <= 3;
""", conn)

,category,item_purchased,purchase_count,rn
0,Accessories,Jewelry,171,1
1,Accessories,Sunglasses,161,2
2,Accessories,Belt,161,3
3,Clothing,Pants,171,1
4,Clothing,Blouse,171,2
5,Clothing,Shirt,169,3
6,Footwear,Sandals,160,1
7,Footwear,Shoes,150,2
8,Footwear,Sneakers,145,3
9,Outerwear,Jacket,163,1


In [32]:
pd.read_sql("""
SELECT
CASE
    WHEN previous_purchases > 5 THEN 'Repeat Buyer'
    ELSE 'Non Repeat Buyer'
END AS buyer_type,
subscription_status,
COUNT(*) AS customer_count
FROM customer
GROUP BY buyer_type, subscription_status;
""", conn)

,buyer_type,subscription_status,customer_count
0,Non Repeat Buyer,No,329
1,Non Repeat Buyer,Yes,95
2,Repeat Buyer,No,2518
3,Repeat Buyer,Yes,958


In [33]:
pd.read_sql("""
SELECT age_group,
       SUM(purchase_amount) AS revenue
FROM customer
GROUP BY age_group
ORDER BY revenue DESC;
""", conn)

,age_group,revenue
0,Young Adult,62143
1,Middle-aged,59197
2,Adult,55978
3,Senior,55763


In [34]:
df.to_csv('customer_behavior_cleaned.csv', index=False)

In [35]:
import os

os.path.exists('customer_behavior_cleaned.csv')

True

In [36]:
df.to_csv('customer_behavior_cleaned.csv', index=False)
from google.colab import files
files.download('customer_behavior_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>